# Task

You will use an LLM API to predict a score for each student answer.

The test data contains:

- `id`: the answer ID
- `question`: the assignment question
- `student_answer`: the answer that must be graded

Your submission file must contain exactly two columns:

- `id`
- `predicted_score`

Each predicted score must be between **0 and 5**.

# Main Part to Edit

You mainly need to edit the `PROMPT_TEMPLATE` variable in the section:

> **PROMPT ENGINEERING**

Do not change the `id` values or the number of rows in the submission file.


## 1. Get an API Key

You can use Google Gemini or OpenAI.

### Option A — Google Gemini

1. Open Google AI Studio.
2. Sign in with your Google account.
3. Select **Get API key** or **Create API key**.
4. Copy your API key.
5. In the Kaggle Notebook, open **Add-ons → Secrets**.
6. Create a secret named:

```text
GEMINI_API_KEY
```

7. Paste your API key into the value field.
8. Enable access for this notebook.

### Option B — OpenAI

1. Open the OpenAI API platform.
2. Create a new API key.
3. Copy your API key.
4. In the Kaggle Notebook, open **Add-ons → Secrets**.
5. Create a secret named:

```text
OPENAI_API_KEY
```

6. Paste your API key into the value field.
7. Enable access for this notebook.

> Never write your API key directly in the code. Never share your API key in a public notebook.


## 2. Install the Required Library

This notebook uses the OpenAI Python library.

Google Gemini also provides an OpenAI-compatible API, so the same library can be used for both providers.


In [1]:
!pip install -q -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.2 MB/s eta 0:00:00a 0:00:01


In [2]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

## 3. Load the Test Set

Add the competition data to your notebook before running the code.

The notebook will automatically search for `Q5_data.csv` inside `/kaggle/input/`.


In [3]:
test_paths = list(Path("/kaggle/input").rglob("Q5_data.csv"))

if not test_paths:
    raise FileNotFoundError(
        "test.csv was not found. Use Add Data to add the competition data."
    )

TEST_PATH = test_paths[0]
print("Using:", TEST_PATH)

test_df = pd.read_csv(TEST_PATH)

required_columns = {"id", "question", "student_answer"}
missing_columns = required_columns - set(test_df.columns)

if missing_columns:
    raise ValueError(
        f"test.csv is missing these columns: {sorted(missing_columns)}"
    )

print("Number of answers to grade:", len(test_df))
test_df.head()

Using: /kaggle/input/competitions/t-1-nthu-summer-camp-0723/Q5_data.csv
Number of answers to grade: 20


,id,question,student_answer
0,q5_001,前提：使用 LSTM 進行數學式的加減乘運算。題目：Why do we need gradi...,To prevent gradient explosion. There is a conn...
1,q5_002,前提：使用 LSTM 進行數學式的加減乘運算。題目：Why do we need gradi...,(LSTM) 有gradient clipping Epoch 30: EM:0.9273 ...
2,q5_003,前提：使用 LSTM 進行數學式的加減乘運算。題目：Why do we need gradi...,防止梯度爆炸 …
3,q5_004,前提：使用 LSTM 進行數學式的加減乘運算。題目：Why do we need gradi...,gradient clipping可以幫助RNN 或 LSTM模型在訓練過程中出現梯度爆炸的...
4,q5_005,前提：使用 LSTM 進行數學式的加減乘運算。題目：Why do we need gradi...,"Keeps optimizer steps in a reasonable range, p..."


## 4. Choose an API Provider

Choose one provider:

```python
PROVIDER = "gemini"
```

or:

```python
PROVIDER = "openai"
```

You also need to enter a model name that is available in your account.

### How to Choose a Model Name

`MODEL_NAME` is the exact name of the AI model that will grade the answers.

You cannot create your own model name. You must use a model name provided by Google Gemini or OpenAI.

The model name must match the API provider that you choose.

---

1. Open the official Gemini/OpenAI model page.
2. Choose a text model that is available for your API key.
3. Copy the model name exactly.
4. Paste it into `MODEL_NAME`.

Example:

```python
PROVIDER = "gemini"
MODEL_NAME = "gemma-4-26b-a4b-it"


In [4]:
# ============================================================
# CHOOSE THE API PROVIDER AND MODEL
# ============================================================

PROVIDER = "gemini"
MODEL_NAME = "gemma-4-26b-a4b-it"

secrets = UserSecretsClient()

if PROVIDER == "gemini":
    API_KEY = secrets.get_secret("GEMINI_API_KEY")
    client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )

elif PROVIDER == "openai":
    API_KEY = secrets.get_secret("OPENAI_API_KEY")
    client = OpenAI(api_key=API_KEY)

else:
    raise ValueError('PROVIDER must be either "gemini" or "openai".')

if MODEL_NAME == "YOUR_MODEL_NAME":
    raise ValueError("Enter MODEL_NAME before running this cell.")

print("API client created for:", PROVIDER)
print("Model:", MODEL_NAME)

API client created for: gemini
Model: gemma-4-26b-a4b-it


## 5. Prompt Engineering

Edit the prompt to help the LLM produce more accurate and consistent scores.

The LLM must return output in this format:

```json
{"score": 3}
```

Do not include Markdown or any text outside the JSON object.


In [5]:
# ============================================================
# THIS IS THE MAIN PART THAT PARTICIPANTS SHOULD EDIT
# ============================================================

PROMPT_TEMPLATE = """
You are a teacher grading a student answer.

Read the question and the student answer. Then give a score from 0 to 5.

Rules:
1. Use only the information provided.
2. A score of 0 means the answer is completely wrong or does not answer the question.
3. A score of 5 means the answer is correct, complete, and well explained.
4. Lower the score when the answer is missing important information, evidence, or clear reasoning.
5. Return exactly one JSON object in this format:
   {{"score": 3}}
6. Do not use Markdown.
7. Do not write anything outside the JSON object.

QUESTION:
{question}

STUDENT ANSWER:
{student_answer}
"""

## 6. Create the API Function

The function below will:

- insert the question and student answer into the prompt;
- call the LLM;
- read the `score` field;
- keep the score between 0 and 10;
- retry if the API has a temporary error.


In [6]:
def parse_score(raw_text: str) -> float:
    if not isinstance(raw_text, str):
        raise TypeError(f"Expected a string, got {type(raw_text).__name__}")

    raw_text = raw_text.strip()
    result = None

    # 1. Try direct parsing first
    try:
        result = json.loads(raw_text)
    except json.JSONDecodeError:
        pass

    # 2. Try greedy search for a JSON block containing "score"
    if not isinstance(result, dict) or "score" not in result:
        # Matches from { to } specifically wrapping a "score" key
        match = re.search(r"\{.*\"score\"\s*:\s*.*\}", raw_text, flags=re.DOTALL)
        if match:
            try:
                result = json.loads(match.group(0))
            except json.JSONDecodeError:
                result = None

    # 3. Fallback: search for any simple JSON object
    if not isinstance(result, dict) or "score" not in result:
        matches = re.findall(r"\{[^{}]*\}", raw_text, flags=re.DOTALL)
        for candidate in matches:
            try:
                parsed = json.loads(candidate)
                if isinstance(parsed, dict) and "score" in parsed:
                    result = parsed
                    break
            except json.JSONDecodeError:
                continue

    if not isinstance(result, dict) or "score" not in result:
        raise ValueError(
            f"No valid JSON object with a 'score' field was found in: {raw_text[:300]}"
        )

    # 4. Safely extract and clamp the score
    try:
        score = float(result["score"])
    except (ValueError, TypeError) as e:
        raise ValueError(
            f"Score value '{result['score']}' cannot be converted to float."
        ) from e

    # Pure Python min/max avoids needing numpy just to clip numbers
    return max(0.0, min(10.0, score))


def grade_one(
    question: str,
    student_answer: str,
    max_retries: int = 3,
) -> float:
    prompt = PROMPT_TEMPLATE.format(
        question=question,
        student_answer=student_answer,
    )

    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a grading system. "
                            "Follow the required JSON format exactly."
                        ),
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
                temperature=0,
            )

            raw_text = response.choices[0].message.content
            return parse_score(raw_text)

        except Exception as error:
            last_error = error
            error_text = str(error)
        
            print(f"Attempt {attempt + 1} failed: {error}")
        
            if attempt < max_retries - 1:
                if "429" in error_text:
                    wait_seconds = 15
                else:
                    wait_seconds = 2 ** attempt
        
                print(
                    f"Waiting {wait_seconds} seconds before retrying..."
                )
                time.sleep(wait_seconds)

    raise RuntimeError(
        f"Could not grade the answer after {max_retries} attempts: "
        f"{last_error}"
    )

## 7. Test One Answer

Always test one answer before grading the full test set.

This helps you check:

- the API key;
- the model name;
- the prompt;
- the JSON output;
- the API quota.


In [ ]:
example = test_df.iloc[0]

example_score = grade_one(
    question=example["question"],
    student_answer=example["student_answer"],
)

print("ID:", example["id"])
print("Predicted score:", example_score)

## 8. Grade the Full Test Set

The notebook saves a checkpoint after each answer.

If the notebook stops, run this cell again to continue from the saved checkpoint.


In [ ]:
CHECKPOINT_PATH = Path("/kaggle/working/prediction_checkpoint.csv")

# Wait between requests to avoid API rate limits.
REQUEST_DELAY = 3

if CHECKPOINT_PATH.exists():
    prediction_df = pd.read_csv(CHECKPOINT_PATH)
    completed_ids = set(prediction_df["id"])
    print(f"Loaded {len(prediction_df)} results from the checkpoint.")
else:
    prediction_df = pd.DataFrame(
        columns=["id", "predicted_score"]
    )
    completed_ids = set()

new_predictions = []

for _, row in test_df.iterrows():
    if row["id"] in completed_ids:
        continue

    score = grade_one(
        question=row["question"],
        student_answer=row["student_answer"],
    )

    new_predictions.append(
        {
            "id": row["id"],
            "predicted_score": score,
        }
    )

    current_predictions = pd.concat(
        [prediction_df, pd.DataFrame(new_predictions)],
        ignore_index=True,
    )

    current_predictions.to_csv(
        CHECKPOINT_PATH,
        index=False,
    )

    print(
        f"[{len(current_predictions)}/{len(test_df)}] "
        f"{row['id']} -> {score}"
    )

    # Wait before sending the next API request.
    time.sleep(REQUEST_DELAY)

prediction_df = pd.read_csv(CHECKPOINT_PATH)
prediction_df.head()

## 9. Create `submission.csv`

The submission file must have exactly these two columns:

```text
id,predicted_score
```


In [ ]:
submission = test_df[["id"]].merge(
    prediction_df[["id", "predicted_score"]],
    on="id",
    how="left",
    validate="one_to_one",
)

if submission["predicted_score"].isna().any():
    missing_ids = submission.loc[
        submission["predicted_score"].isna(),
        "id",
    ].tolist()

    raise ValueError(
        f"Predictions are missing for these IDs: {missing_ids[:10]}"
    )

if submission["id"].duplicated().any():
    raise ValueError("The submission contains duplicate IDs.")

if not submission["predicted_score"].between(0, 10).all():
    raise ValueError("Some scores are outside the range 0 to 10.")

SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False)

print("Created:", SUBMISSION_PATH)
print("Shape:", submission.shape)

submission.head()

## 10. Submit Your Results

### Option A - Download the file and then upload
1. Open the **Output** section of the Kaggle Notebook.
2. Download `submission.csv`.
3. Open the competition page.
4. Select **Submit Predictions**.
5. Upload `submission.csv`.

### Option B - Save the notebook and submit with notebook
1. Use the **Save Version** button on the upper-right of the Kaggle Notebook.
2. Select `Save & Run All (Commit)` (if haven't executed) or `Quick Save` (if have already executed).
3. Open the competition page.
4. Select **Submit Predictions**.
5. Upload by `Notebook`, and select the latest version of your notebook (output file: `submission.csv`).

**Reminder**: Do not change the `id` column after creating the file.
